In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from tkinter.filedialog import  askopenfilenames,askopenfilename
import  plotly.graph_objects as go
import os
import numpy as np
%matplotlib widget

def plot_spectra_from_csv(file_paths, _norm=False, _wavelength_range=None):
    fig = go.Figure()

    for file_path in file_paths:
        data = pd.read_csv(file_path, header=None)
        num_spectra = data.shape[1] // 2  # Number of spectra in the file

        for i in range(num_spectra):
            wavelength = pd.to_numeric(data.iloc[2:, 2 * i], errors='coerce')
            absorbance = pd.to_numeric(data.iloc[2:, 2 * i + 1], errors='coerce')
            label = f"{file_path.split('/')[-1]} - {data.iloc[0, 2 * i]}"  # Unique label per file
            if _wavelength_range:
                mask = (wavelength >= _wavelength_range[0]) & (wavelength <= _wavelength_range[1])
                wavelength = wavelength[mask]
                absorbance = absorbance[mask]
            y_data = absorbance / np.max(absorbance) if _norm else absorbance
            fig.add_trace(go.Scatter(
                x=wavelength, 
                y=y_data, 
                mode='lines', 
                name=label,
                line=dict(width=3)  # Line width
            ))

    fig.update_layout(
        xaxis_title='Wavelength (nm)',
        yaxis_title='Absorbance',
        legend_title='Spectrum',
        template='plotly_white',
        width=1000,
        height=600,
        font=dict(size=10),  # Label font size (title, legend, etc.)
        xaxis=dict(
            tickfont=dict(size=20),  # X tick size
            title_font=dict(size=25),
        ),
        yaxis=dict(
            tickfont=dict(size=20),  # Y tick size
            title_font=dict(size=25),
            range=[0, 1]
        )
    )

    fig.show()


def read_and_plot_txt(file_paths, normalize=False):
    fig = go.Figure()

    for file_path in file_paths:
        data = []
        with open(file_path, 'r') as file:
            lines = file.readlines()
            for line in lines[1:]:  # Skip the header line
                parts = line.split()
                if len(parts) == 2:
                    pixel = int(parts[0])
                    value = float(parts[1])
                    data.append((pixel, value))

        pixels, values = zip(*data)

        if normalize:
            values = [v / max(values) for v in values]  # Normalize the values

        # Extract the file name without the extension
        file_name = os.path.splitext(os.path.basename(file_path))[0]

        fig.add_trace(go.Scatter(
            x=pixels,
            y=values,
            mode='lines',
            name=file_name  # Use the file name as the label
        ))

    fig.update_layout(
        xaxis_title='Pixel',
        yaxis_title='OD' if not normalize else 'Normalized OD',
        legend_title='File',
        template='plotly_white',
        width=1000,
        height=600
    )

    fig.show()

def plot_difference_spectra(file_path1, file_path2, spectrum_index1, spectrum_index2):
    # Load the CSV files
    data1 = pd.read_csv(file_path1, header=None)
    data2 = pd.read_csv(file_path2, header=None)

    # Extract wavelength and absorbance data for the selected spectra
    wavelength1 = pd.to_numeric(data1.iloc[2:, 2 * spectrum_index1], errors='coerce')
    absorbance1 = pd.to_numeric(data1.iloc[2:, 2 * spectrum_index1 + 1], errors='coerce')

    wavelength2 = pd.to_numeric(data2.iloc[2:, 2 * spectrum_index2], errors='coerce')
    absorbance2 = pd.to_numeric(data2.iloc[2:, 2 * spectrum_index2 + 1], errors='coerce')

    # Ensure both spectra have the same wavelength range
    if not wavelength1.equals(wavelength2):
        raise ValueError("The wavelength ranges of the two spectra do not match.")

    # Compute the difference in absorbance
    difference = absorbance1 - absorbance2

    # Plotting the difference spectrum using Plotly
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=wavelength1, y=difference, mode='lines', name=f'Difference Spectrum ({data1.iloc[0, 2 * spectrum_index1]} - {data2.iloc[0, 2 * spectrum_index2]})', line=dict(color='red')))

    fig.update_layout(
        xaxis_title='Wavelength (nm)',
        yaxis_title='Absorbance Difference',
        legend_title='Spectra',
        template='plotly_white',
        width=1000,  # Increase the width of the plot
        height=600,  # Increase the height of the plot
        yaxis=dict(range=[-12, 12])  # Set y-axis limit to max 12
    )
    fig.show()

def smooth_data_average_window(data, window_size):
    # Calculate rolling mean with specified window size
    windows = data.rolling(spectral= window_size ,center = True, min_periods=1)
    moving_average = windows.mean()
    return moving_average

# Open all the spectrum in the files

In [ ]:
# Open file dialog to select CSV files
file_paths = askopenfilenames(filetypes=[("CSV files", "*.csv")], title="Select the UV-vis data file")

# Plot spectra from the selected files
plot_spectra_from_csv(file_paths,_norm = True,_wavelength_range=[380,700])


In [ ]:
plot_spectra_from_csv(file_paths,_norm = True,_wavelength_range=[380,700])

# PLot from .txt file (Copied from the LabView TA test Window)

In [ ]:
# Open file dialog to select text files
file_paths = askopenfilenames(filetypes=[("txt files", "*.txt")], title="Select the txt data file")

read_and_plot_txt(file_paths,normalize= False)

# Spectrum Difference - Work in progress

In [ ]:
import numpy as np


# Function to read the spectrum from a file
def read_spectrum(filename):
    wavelengths, intensities = [], []
    with open(filename, 'r') as file:
        for line in file:
            if ':' in line:
                continue  # Skip the header line
            parts = line.strip().split()
            if len(parts) == 2:
                wavelengths.append(float(parts[0]))
                intensities.append(float(parts[1]))
    return np.array(wavelengths), np.array(intensities)

# Read the two files
wavelengths1, intensities1 = read_spectrum('/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/250319_Control4/250320_WLSpectrum_Cu(dsbp)_1mmPL.txt')
wavelengths2, intensities2 = read_spectrum('/Users/vard/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/New Cu/250319_Control4/250320_WLSpectrum_Cu(dsbp)_NoPump_1mmPL.txt')

# Compute the difference
difference = intensities2 - intensities1

# Create figure
fig = go.Figure()

# Add first spectrum
fig.add_trace(go.Scatter(x=wavelengths1, y=intensities1, mode='lines', name='Pump On', opacity=0.3))

# Add second spectrum
fig.add_trace(go.Scatter(x=wavelengths2, y=intensities2, mode='lines', name='Pump off', opacity =0.3))

# Add difference
fig.add_trace(go.Scatter(x=wavelengths1, y=difference, mode='lines', name='Difference'))

# Layout adjustments
fig.update_layout(xaxis_title='Wavelength (nm)',
                  yaxis_title='Intensity')

# Show plot
fig.show()